# M1.6 · Mensajes multimodales 🖼️

**Lección oficial:** *Multimodal messages* · **Notebook original:** `module-1/1.4_multimodal_messages.ipynb`

## 🎯 Qué vas a aprender
- Qué significa **multimodal**: texto + imagen (+ audio).
- Cómo armar un mensaje con **bloques de contenido** (`content=[...]`).
- Cómo mandar una **imagen** convertida a **base64**.

## 🧸 Explicado sencillo
Hasta ahora el agente solo **leía**. Un modelo multimodal también puede **mirar** 👀 (imágenes) y a veces **escuchar** 👂 (audio).

Un mensaje multimodal es como una **carta con fotos adjuntas**: un bloque de texto + un bloque de imagen.

Aquí usamos **gemma4**, que sí ve imágenes (qwen3 no).

## 🛠️ Paso 0 · Preparar el notebook

Esta celda hace tres cosas por ti:
1. Encuentra la carpeta principal del repo (donde vive `local_model.py`).
2. Lee tu archivo `.env` (tus llaves y la dirección del servidor) **sin mostrarlas**.
3. Revisa el **semáforo** del servidor Ollama, que es **compartido**:
   - 🟢 libre → adelante.
   - 🟡 hay un modelo cargado → si fuiste tú hace un rato, sigue; si no, espera.
   - 🔴 sin conexión → revisa `OLLAMA_BASE_URL` en tu `.env`.

> Corre las celdas **en orden, de arriba hacia abajo** (Shift + Enter). El número `[n]` a la izquierda te dice en qué orden se ejecutaron.

In [ ]:
# 🟢 ADAPTADO LOCAL: preparar entorno (ver ayuda.py)
from ayuda import preparar, ver, semaforo
preparar()

from local_model import get_model   # se importa DESPUÉS de cargar el .env

## 1 · Bloques de contenido (solo texto)
En vez de `content="texto"`, el contenido puede ser una **lista de bloques**. Cada bloque dice su tipo: `"text"`, `"image"`, `"audio"`…

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre
# agent = create_agent(model="gpt-5-nano", system_prompt=...)

# 🟢 ADAPTADO LOCAL: gemma4 entiende texto e imágenes
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model=get_model("gemma4:latest"),
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)

question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"},
])
response = agent.invoke({"messages": [question]})
print(response["messages"][-1].content)

## 2 · Preparar una imagen: base64 🖼️➡️🔤
Los modelos reciben las imágenes como **texto codificado** llamado **base64**.

🧸 Es como convertir una foto en una larguísima fila de letras para poder mandarla por "mensaje de texto", y del otro lado la vuelven a armar.

El curso original sube la imagen con un botón (`FileUpload`). Aquí la leemos directo del archivo `moon.png` que viene en el repo: más fácil.

In [ ]:
# 🟢 ADAPTADO LOCAL: leer la imagen del repo en vez del botón de subida
import base64
import pathlib
from IPython.display import Image, display

ruta = pathlib.Path("../module-1/resources/moon.png")
display(Image(filename=str(ruta), width=300))          # la vemos nosotros

img_b64 = base64.b64encode(ruta.read_bytes()).decode("utf-8")
print("Largo del texto base64:", len(img_b64), "caracteres")
print("Primeros 60:", img_b64[:60], "…")

## 3 · Mandar texto + imagen 📨
Un bloque de texto (la pregunta) y un bloque de imagen (la foto en base64). Le decimos su tipo con `mime_type`.

⏳ Las imágenes tardan más en CPU (1 a 3 minutos). Paciencia.

In [ ]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this capital"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"},
])

response = agent.invoke({"messages": [multimodal_question]})
print(response["messages"][-1].content)

🤔 **Pregunta para pensar:** ¿describió cosas que **sí** están en la imagen, o inventó? Compara con la foto de arriba.

## 4 · ¿Y el audio? 🎤 (opcional)
El curso también manda **audio** grabado con el micrófono. Con modelos locales es más complicado: hace falta grabar a 16 kHz y un truco para que gemma4 lo acepte.

Si te da curiosidad, mira el notebook original `module-1/1.4_multimodal_messages.ipynb` (sección *Audio input*) y la nota `docs/module-1/M1.6-multimodal.md`. **No es necesario para el certificado.**

## ✍️ Tu turno
1. Pon cualquier imagen `.png` tuya en la carpeta y cambia `ruta` para que apunte a ella.
2. Pregunta en español: *"¿Qué ves en esta imagen? Descríbela en 3 frases."*
3. Reto: pídele que invente una **historia** de ciencia ficción basada en tu imagen.

## ✅ Mini quiz
1. ¿Qué significa *multimodal*?
2. ¿Qué forma tiene `content` en un mensaje multimodal?
3. ¿Para qué se convierte la imagen a base64?
4. ¿Por qué aquí usamos gemma4 y no qwen3?

<details><summary>Respuestas</summary>

1. Que el modelo entiende más de un tipo de dato: texto, imagen, audio.
2. Una lista de bloques, cada uno con su `type`.
3. Para enviarla como texto dentro del mensaje.
4. Porque gemma4 tiene visión; qwen3 solo entiende texto.
</details>

## 🧠 Resumen en una línea
**Multimodal = carta con fotos adjuntas. `content` = lista de bloques. Imagen → base64 → bloque `"image"`.**